# Part B - Custom Retrieval System (Fresh Start)
**Student:** Dabone Abdoul Latif  
**Index:** 10012200015  

This notebook handles Tasks 1 through 5 in one clean process.

In [ ]:
print("Step 1: Importing libraries...")
import json
import os
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

print("Step 2: Loading chunks...")
with open('chunks/csv_chunks.json', 'r') as f:
    csv_chunks = json.load(f)
with open('chunks/pdf_chunks.json', 'r') as f:
    pdf_chunks = json.load(f)
all_chunks = csv_chunks + pdf_chunks
texts = [c['text'] for c in all_chunks]
print(f"--- Loaded {len(all_chunks)} chunks.")

print("Step 3: Loading embedding model (This may take 1-2 minutes for the first download)... ")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("--- Model loaded successfully.")

print("Step 4: Generating embeddings... ")
embeddings = model.encode(texts, show_progress_bar=True)

print("Step 5: Creating FAISS index...")
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype('float32'))
os.makedirs('indexes', exist_ok=True)
faiss.write_index(index, 'indexes/rag_index.faiss')
print(f"--- FAISS index saved with {index.ntotal} vectors.")

print("Step 6: Setting up BM25 for Hybrid Search...")
tokenized_corpus = [text.lower().split() for text in texts]
bm25 = BM25Okapi(tokenized_corpus)
print("--- Hybrid Search Ready.")

## Task 3, 4 & 5: Search and Failure Fix Demo

In [ ]:
def hybrid_search(query, k=5, alpha=0.5):
    # Vector Search
    query_vector = model.encode([query]).astype('float32')
    distances, indices = index.search(query_vector, k*2)
    
    # BM25 Search
    bm25_scores = bm25.get_scores(query.lower().split())
    max_bm25 = max(bm25_scores) if max(bm25_scores) > 0 else 1
    
    combined = []
    for i, chunk in enumerate(all_chunks):
        # Vector similarity (convert distance to similarity)
        v_score = 0
        for rank, idx in enumerate(indices[0]):
            if idx == i:
                v_score = 1 / (1 + distances[0][rank])
                break
        
        # Hybrid combine
        score = (alpha * v_score) + ((1 - alpha) * (bm25_scores[i] / max_bm25))
        if score > 0: combined.append({'chunk': chunk, 'score': score})
            
    return sorted(combined, key=lambda x: x['score'], reverse=True)[:k]

# DEMO FAILURE FIX
query = "What is the targeted inflation rate for 2025?"
print(f"Query: {query}\n")

results = hybrid_search(query, k=1)
print("Top Retrieval Result:")
print(results[0]['chunk']['text'])